# P3. Build a Research Agent From Scratch (Capstone)

**Tier:** Projects (capstone for Tier 4)
**Estimated time:** 90 minutes
**Prerequisites:** 18, 19, 20, 21, 22, 23
**Priority:** 🟡 Important — first capstone synthesis of Tier 4 — worth doing before P4's larger system. *If skipped, revisit when:* before attempting P4.
**Source material:** synthesizes the full Agent Engineering tier (@sairahul1 harness; @humzaakhalid loops; @akasheth_ multi-agent)

## What You'll Learn
- How to assemble one real agent from every Tier 4 idea: the raw loop, harness, LangGraph, LangSmith, context management, and verification loops
- How to enforce a tool budget and an adversarial self-check so the agent is *reliable*, not just functional
- How the three lenses — **Claude SDK**, **LangGraph**, **LangSmith** — fit together in one system

## Why This Matters
This is the graduation project. Everything in Tier 4 was a piece; here they become a working research agent that takes a question, gathers sources, synthesizes findings, *verifies its own work*, and writes a structured report to disk. Build this and you've built the skeleton of every serious agent product.


## The architecture: five stages on a graph

Our research agent is a **LangGraph** (notebook 20) whose nodes are the stages of doing research, with each stage drawing on a different Tier 4 idea:

```
              ┌─────────┐
  question ──▶│  plan   │  break the question into sub-questions
              └────┬────┘
                   ▼
              ┌─────────┐
              │ research│  a RAW Claude agent loop (nb 19) with a web_search tool,
              └────┬────┘  budget-enforced (nb 19) so it can't search forever
                   ▼
              ┌─────────┐
              │synthesize│ write a structured draft from the gathered sources
              └────┬────┘
                   ▼
              ┌─────────┐   adversarial checker (nb 23): a SEPARATE call whose
              │ verify  │◀┐ job is to find flaws — loops back until approved
              └────┬────┘ │ or the revision budget runs out
                   │ approved? ──no──▶ revise ──┘
                   ▼ yes
              ┌─────────┐
              │  write  │  save a structured report to disk
              └─────────┘
```

Surrounding the graph:
- **Harness** (notebook 21): a CLAUDE.md-style brief and a JSON progress tracker bootstrap every run.
- **Context management** (notebook 22): sources are *written* to state and *selected* into each prompt, not all crammed into one window.
- **LangSmith** (notebook 21): the whole run is traceable so you can inspect it afterward.

We keep the agent self-contained and offline-capable: `web_search` queries a small mock knowledge base, so the notebook runs deterministically without internet (only Claude calls need the key).


In [ ]:
import os, json, pathlib, tempfile
from typing import TypedDict, Annotated
import operator
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
HAS_LANGSMITH = bool(os.environ.get("LANGSMITH_API_KEY"))
TEACH_MODEL = "claude-haiku-4-5-20251001"   # production default: claude-opus-4-8

if HAS_ANTHROPIC:
    import anthropic
    client = anthropic.Anthropic()
    print("Anthropic ready — the agent will run live.")
else:
    client = None
    print("No ANTHROPIC_API_KEY — the graph is built and explained, node execution is skipped.")

def ask(prompt, system="You are a precise research assistant.", max_tokens=400, temperature=0.3):
    if not HAS_ANTHROPIC:
        return "[skipped: no ANTHROPIC_API_KEY]"
    try:
        msg = client.messages.create(model=TEACH_MODEL, max_tokens=max_tokens, system=system,
                                      temperature=temperature, messages=[{"role": "user", "content": prompt}])
        return msg.content[0].text
    except Exception as e:
        return f"[skipped: {type(e).__name__}: {str(e)[:120]}]"

WORKDIR = pathlib.Path(tempfile.mkdtemp(prefix="p3_agent_"))
print("Agent working directory:", WORKDIR)


## Part 1 — The harness (notebook 21)

A brief that defines the agent's role and output contract, plus a JSON tracker that records progress through the run. Bootstrapped once by `init_session`.


In [ ]:
(WORKDIR / "AGENT.md").write_text("""# Research Agent Brief

## Role
You research a question using the web_search tool, then write a structured, sourced report.

## Output contract (the report MUST have)
- A one-paragraph Summary
- 2-4 Key Findings, each tied to a source
- A Caveats section noting what is uncertain or unsupported
- Never state a claim you did not find via web_search.
""")

PROGRESS_FILE = WORKDIR / "progress.json"

def init_session(directory):
    """Bootstrap a run: load the brief, reset the progress tracker."""
    brief = (pathlib.Path(directory) / "AGENT.md").read_text()
    PROGRESS_FILE.write_text(json.dumps(
        {"stage": "init", "searches_used": 0, "verify_rounds": 0}, indent=2))
    system_prompt = ("You are a research agent for this project. Follow its brief exactly.\n\n"
                     f"<brief>\n{brief}\n</brief>")
    return system_prompt

def update_progress(**kwargs):
    p = json.loads(PROGRESS_FILE.read_text())
    p.update(kwargs)
    PROGRESS_FILE.write_text(json.dumps(p, indent=2))
    return p

SYSTEM_PROMPT = init_session(WORKDIR)
print("Session initialized. Brief loaded:", len(SYSTEM_PROMPT), "chars")
print("Progress:", json.loads(PROGRESS_FILE.read_text()))


## Part 2 — Tools and the raw research sub-agent (notebooks 18-19)

`web_search` is a mock knowledge base so the agent runs offline. The research node is a **raw Anthropic agent loop** (notebook 19) with a **tool-call budget** (notebook 19's exercise 3): once the budget is spent, we withhold tools to force the agent to conclude.


In [ ]:
# Mock knowledge base — the agent's searchable "web".
_KB = {
    "multi-agent benefits": "Multi-agent systems parallelize work and let specialists focus, improving quality on complex tasks (source: akasheth_).",
    "multi-agent costs": "Multi-agent systems add orchestration complexity, latency, and token cost; start with 2 agents (source: akasheth_).",
    "single agent": "A single agent in a loop handles most tasks with far less coordination overhead (source: nb19).",
    "when multi-agent": "Use multiple agents when work is parallelizable or needs distinct expertise; otherwise prefer one agent (source: akasheth_).",
}

def web_search(query):
    q = query.lower()
    hits = [v for k, v in _KB.items() if any(w in q for w in k.split())]
    return " ".join(hits) if hits else "No results found (mock index)."

WEB_SEARCH_SCHEMA = {
    "name": "web_search",
    "description": "Search the web for information on a topic. Returns source snippets.",
    "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
}

def research_subagent(subquestions, max_searches=3):
    """Raw Claude agent loop with a search budget. Returns collected source snippets."""
    if not HAS_ANTHROPIC:
        return ["[skipped: no ANTHROPIC_API_KEY]"]
    goal = ("Research these sub-questions using web_search, then stop:\n- " + "\n- ".join(subquestions))
    messages = [{"role": "user", "content": goal}]
    collected, searches = [], 0
    for _ in range(max_searches + 2):
        over_budget = searches >= max_searches
        kwargs = dict(model=TEACH_MODEL, max_tokens=500, system=SYSTEM_PROMPT, messages=messages)
        if not over_budget:
            kwargs["tools"] = [WEB_SEARCH_SCHEMA]      # withholding tools = forced conclusion (nb19)
        resp = client.messages.create(**kwargs)
        if resp.stop_reason != "tool_use":
            break
        messages.append({"role": "assistant", "content": resp.content})
        results = []
        for b in resp.content:
            if b.type == "tool_use":
                searches += 1
                out = web_search(**b.input)
                collected.append(out)
                results.append({"type": "tool_result", "tool_use_id": b.id, "content": out})
        messages.append({"role": "user", "content": results})
    update_progress(searches_used=searches)
    return collected

if HAS_ANTHROPIC:
    demo = research_subagent(["What are the benefits of multi-agent systems?"], max_searches=2)
    print(f"Collected {len(demo)} source snippet(s); searches used:",
          json.loads(PROGRESS_FILE.read_text())["searches_used"])


## Part 3 — The orchestration graph (notebooks 20 + 23)

Now we wire the five stages into a LangGraph, with the **verify → revise** cycle as a conditional edge that loops until the checker approves or the revision budget runs out.


In [ ]:
from langgraph.graph import StateGraph, START, END

class ResearchState(TypedDict):
    question: str
    subquestions: list
    sources: Annotated[list, operator.add]   # reducer: research appends sources
    draft: str
    review: str
    approved: bool
    verify_rounds: int

def plan_node(state):
    out = ask(f"Break this question into 2 concrete sub-questions, one per line:\n{state['question']}",
              max_tokens=120)
    subs = [line.lstrip("-0123456789. ").strip() for line in out.splitlines() if line.strip()][:2]
    return {"subquestions": subs or [state["question"]]}

def research_node(state):
    sources = research_subagent(state["subquestions"], max_searches=3)
    return {"sources": sources}

def synthesize_node(state):
    src = "\n".join(f"- {s}" for s in state["sources"])
    draft = ask(f"Write a report answering: {state['question']}\n\nUse ONLY these sources:\n{src}\n\n"
                "Format: Summary; Key Findings (each tied to a source); Caveats.",
                system=SYSTEM_PROMPT, max_tokens=500)
    return {"draft": draft}

def verify_node(state):
    """Adversarial checker (nb23): a separate call hunting for unsupported claims."""
    review = ask(f"Question: {state['question']}\nSources:\n{state['sources']}\n\nReport:\n{state['draft']}\n\n"
                 "You are a strict reviewer. If every claim is supported by the sources and the format is "
                 "followed, reply exactly 'APPROVED'. Otherwise name the single worst problem.",
                 system="You are a ruthless reviewer. Do not be agreeable.", max_tokens=150, temperature=0.0)
    approved = "APPROVED" in review.upper()
    rounds = state["verify_rounds"] + 1
    update_progress(verify_rounds=rounds, stage="verify")
    return {"review": review, "approved": approved, "verify_rounds": rounds}

def revise_node(state):
    fixed = ask(f"Revise this report to fix the reviewer's issue.\nReport:\n{state['draft']}\n\n"
                f"Reviewer said: {state['review']}", system=SYSTEM_PROMPT, max_tokens=500)
    return {"draft": fixed}

def write_node(state):
    report_path = WORKDIR / "report.md"
    report_path.write_text(f"# Research Report\n\n**Question:** {state['question']}\n\n{state['draft']}\n")
    update_progress(stage="done")
    return {}

def after_verify(state):
    # Loop back to revise if not approved and we still have budget; else write the report.
    if state["approved"] or state["verify_rounds"] >= 2:
        return "write"
    return "revise"


In [ ]:
graph = StateGraph(ResearchState)
graph.add_node("plan", plan_node)
graph.add_node("research", research_node)
graph.add_node("synthesize", synthesize_node)
graph.add_node("verify", verify_node)
graph.add_node("revise", revise_node)
graph.add_node("write", write_node)

graph.add_edge(START, "plan")
graph.add_edge("plan", "research")
graph.add_edge("research", "synthesize")
graph.add_edge("synthesize", "verify")
graph.add_conditional_edges("verify", after_verify, {"revise": "revise", "write": "write"})
graph.add_edge("revise", "verify")     # revise loops back to the checker
graph.add_edge("write", END)
research_agent = graph.compile()

print("Graph compiled. Nodes:", [n for n in research_agent.get_graph().nodes if not n.startswith("__")])
print("Edges:", [(e.source, e.target) for e in research_agent.get_graph().edges])


## Part 4 — Run it end-to-end, traced with LangSmith (notebook 21)

We wrap the whole invocation in `@traceable` so the run is recorded in LangSmith (if a key is set), then run the agent and read the report it wrote to disk.


In [ ]:
def run_research_agent(question):
    initial = {"question": question, "subquestions": [], "sources": [],
               "draft": "", "review": "", "approved": False, "verify_rounds": 0}
    return research_agent.invoke(initial)

if HAS_LANGSMITH:
    os.environ.setdefault("LANGSMITH_TRACING", "true")
    try:
        from langsmith import traceable
        run_research_agent = traceable(run_type="chain", name="p3_research_agent")(run_research_agent)
        print("LangSmith tracing enabled — this run will appear in your dashboard.")
    except Exception as e:
        print(f"LangSmith tracing skipped (non-fatal): {type(e).__name__}")
else:
    print("No LANGSMITH_API_KEY — running untraced.")

if HAS_ANTHROPIC:
    final_state = run_research_agent("When should you use a multi-agent system instead of a single agent?")
    print("\nFinal progress:", json.loads(PROGRESS_FILE.read_text()))
    print("\n" + "=" * 60)
    print((WORKDIR / "report.md").read_text())
else:
    print("  [skipped: no ANTHROPIC_API_KEY] — the graph above is fully built and inspectable.")


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# Map each pipeline stage to the Tier 4 notebook it draws on.
stages = ["plan", "research", "synthesize", "verify", "write"]
draws_on = ["nb20 graph", "nb19 raw loop\n+ budget", "nb22 context", "nb23 adversarial", "nb21 harness"]
plt.figure(figsize=(9, 3.5))
plt.bar(stages, [1]*5, color=["#4C72B0", "#C44E52", "#55A868", "#DD8452", "#8172B3"])
for i, d in enumerate(draws_on):
    plt.text(i, 0.5, d, ha="center", va="center", color="white", fontsize=8)
plt.title("Each stage of the capstone draws on a different Tier 4 notebook")
plt.yticks([]); plt.tight_layout(); plt.show()


*The capstone isn't new material — it's the whole tier composed into one system: a raw loop wrapped in a graph, kept honest by an adversarial checker, bootstrapped by a harness, and traced end-to-end.*


## Exercises


In [ ]:
# Exercise 1 (Warm-up): Tighten the search budget
# Task: Re-run the agent with max_searches=1 in research_subagent. Does the report quality or the
#       Caveats section change when the agent is forced to gather fewer sources?
# Hint: A tighter budget means less evidence — a good agent should say MORE in Caveats, not invent
#       claims. Check whether yours does.

# YOUR CODE HERE


In [ ]:
# Exercise 2 (Apply): Add a third tool
# Task: Give the research sub-agent a second tool, `lookup_definition(term)`, backed by a small
#       glossary dict. Confirm the raw loop handles two tools with no change to the loop itself.
# Hint: Add a schema + an entry in a tools dict, exactly like notebook 19 — the loop is tool-agnostic.

# YOUR CODE HERE


In [ ]:
# Exercise 3 (Extend): Make verification stricter and watch the loop work
# Task: Strengthen verify_node's reviewer prompt to also require AT LEAST 3 Key Findings. Run the
#       agent and inspect progress.json — how many verify_rounds did it take to satisfy the
#       stricter contract (capped at 2)?
# Hint: This is the adversarial loop from notebook 23 doing its job — a stricter checker forces more
#       revision rounds, which is exactly the reliability/cost tradeoff you're tuning.

# YOUR CODE HERE


<details>
<summary>Show solutions</summary>

```python
# Exercise 1
def research_subagent_tight(subquestions):
    return research_subagent(subquestions, max_searches=1)
# Swap it into research_node (or call directly). With less evidence a well-behaved agent leans on
# the Caveats section instead of fabricating — verify by reading the new report.md.

# Exercise 2
GLOSSARY = {"orchestration": "Coordinating multiple agents/steps toward a goal.",
            "reducer": "A function that merges parallel state updates in LangGraph."}
def lookup_definition(term): return GLOSSARY.get(term.lower(), "Term not in glossary.")
DEF_SCHEMA = {"name": "lookup_definition", "description": "Define a technical term.",
              "input_schema": {"type": "object", "properties": {"term": {"type": "string"}}, "required": ["term"]}}
TOOLS = {"web_search": web_search, "lookup_definition": lookup_definition}
# In the loop, dispatch via TOOLS[b.name](**b.input) and pass tools=[WEB_SEARCH_SCHEMA, DEF_SCHEMA].

# Exercise 3
def verify_node_strict(state):
    review = ask(f"Question: {state['question']}\nSources:\n{state['sources']}\nReport:\n{state['draft']}\n"
                 "Strict review: reply 'APPROVED' only if every claim is sourced AND there are at least "
                 "3 Key Findings; else name the worst gap.",
                 system="You are a ruthless reviewer.", max_tokens=150, temperature=0.0)
    approved = "APPROVED" in review.upper()
    rounds = state["verify_rounds"] + 1
    update_progress(verify_rounds=rounds)
    return {"review": review, "approved": approved, "verify_rounds": rounds}
# Rebuild the graph with verify_node_strict, run, then: json.loads(PROGRESS_FILE.read_text())["verify_rounds"]
```
</details>


## Key Takeaways
- A production agent is a *composition*: a raw loop (nb19) wrapped in a graph (nb20), guarded by an adversarial checker (nb23), bootstrapped by a harness (nb21), with managed context (nb22).
- The three lenses fit together: **Claude SDK** runs the tool loop, **LangGraph** orchestrates the stages, **LangSmith** traces the whole run.
- A tool budget and an adversarial verify→revise cycle are what make the agent *reliable* — without them it's merely functional.
- The output contract (the AGENT.md brief) plus a checker that enforces it is how you stop an agent from inventing claims.
- You can now build the skeleton of any serious agent: plan → act (budgeted) → synthesize → verify → deliver.

## What's Next
You've completed the Agent Engineering tier and its capstone. The natural next step is **Tier 5 — Evaluation & Production** (notebooks 24-27): how to measure whether agents like this one are actually good, with LLM-as-judge and benchmark hygiene — the LangSmith traces you generated here are exactly where that evaluation begins.
